<h1 align=center><font size = 5>Datenanalyse mithilfe von SQLite und Python - Chicago_Crime_Data</font></h1>


# Einführung:

Dieses Projekt ist in 3 Aufgaben eingegliedert mit dem Sinn, Daten mithilfe von SQLite in Jupyter Notebook abzufragen:

1.  Zuerst werden die drei unterschiedlichen Chicago-Datensätze heruntergeladen
2.  Anschließend werden die Datensätze in 3 Tabellen in eine gemeinsame SQLite-Datenbank hochgeladen
3.  Zum Schluss werden wichtige Daten anhand von SQLite in Python abgefragt

Obwohl dieses Projekt im Rahmen einer Abschlussprüfung entstanden ist, dient es an dieser Stelle primär als Leitfaden. Ziel ist es, den methodischen Ablauf einer Datenanalyse mit Python und SQLite verständlich zu veranschaulichen. Alle hier vorhandenen Datensätze stammen aus dem Online-Kurs "Databases and SQL for Data Science with Python".


## Die Drei Chicago-Datensätze

Die jeweiligen Datensätze lauten:

1.  Socioeconomic Indicators in Chicago
2.  Chicago Public Schools
3.  Chicago Crime Data

### 1. Socioeconomic Indicators in Chicago

Dieser Datensatz enthält für jeden Stadtteil von Chicago eine Auswahl von sechs sozialen und wirtschaftlichen Merkmalen, die wichtig für die öffentliche Gesundheit sind. Außerdem enthält er einen Elends-Index (Hardship Index) für die Jahre 2008 bis 2012.

### 2. Chicago Public Schools

Dieser Datensatz zeigt die Leistungsdaten aller Schulen, aus denen die Schulberichte der Chicago Public Schools (CPS) für das Schuljahr 2011–2012 erstellt wurden. Die Daten stammen aus dem Datenportal der Stadt Chicago.

### 3. Chicago Crime Data

Dieser Datensatz enthält alle gemeldeten Straftaten in Chicago von 2001 bis heute. Die einzigen Ausnahmen sind Morde, bei denen die Opfer einzeln erfasst werden. Zudem fehlen aus Aktualitätsgründen immer die Daten der letzten sieben Tage.

### Download

Da die zu analysierenden Daten im Internet häufig als CSV-Dateien (Comma-Separated Values) vorliegen, nutze ich diese hier ebenfalls als Basis.

Zuerst habe ich die Datensätze aus dem Online-Kurs "Databases and SQL for Data Science with Python" heruntergeladen:

* Chicago Census Data

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01

* Chicago Public Schools

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01

* Chicago Crime Data

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCrimeData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01

Die hier verlinkten Versionen sind Auszüge der Originaldatensätze. Dabei wurden einige Spaltennamen angepasst, um sie datenbankfreundlicher zu gestalten und die anschließende Analyse zu erleichtern.


## Vorgang

### 1. pandas- und ipython-Paket herunterladen

Bevor wir mit der eigentlichen Datenanalyse starten können, müssen wir unsere Arbeitsumgebung vorbereiten. Im folgenden Schritt installiere und konfiguriere ich die notwendigen Python-Bibliotheken. Damit stellen wir sicher, dass wir Datensätze verarbeiten und SQL-Abfragen direkt im Notebook ausführen und übersichtlich darstellen können.

In [1]:
!pip install pandas
!pip install ipython-sql prettytable 
# Dient dazu, SQL-Befehle direkt in Python-Zellen zu schreiben und die Ergebnisse als übersichtliche Tabellen anzuzeigen.
import prettytable

prettytable.DEFAULT = 'DEFAULT'

### 2. Daten in datenbasierte Tabellen speichern

Um die Daten mithilfe von SQL analysieren zu können, müssen sie zuerst in eine SQLite-Datenbank geladen werden. Dazu erstelle ich im Folgenden drei Tabellen mit der folgenden Struktur:

1.  **CENSUS_DATA**
2.  **CHICAGO_PUBLIC_SCHOOLS**
3.  **CHICAGO_CRIME_DATA**


Im nächsten Schritt lade ich die Bibliotheken pandas und sqlite3 in die Laufzeitumgebung und stelle eine Verbindung zur lokalen Datenbank `FinalDB.db` her.


In [2]:
import pandas as pd
import csv, sqlite3

con = sqlite3.connect("FinalDB.db") # eine leere Datenbank namens FinalDB wird erstellt.
cur = con.cursor() 

Um SQL-Befehle direkt in den Zellen dieses Notebooks ausführen zu können, aktiviere ich nun das SQL-Magic-Modul (`%load_ext sql`).

In [3]:
%load_ext sql

Nun verwende ich Pandas, um die zuvor verlinkten CSV-Dateien direkt aus dem Internet in DataFrames einzulesen. Anschließend nutze ich diese Datenstrukturen, um die entsprechenden Zieltabellen in unserer Datenbank `FinalDB.db` zu erstellen und zu befüllen.


In [4]:
df_census_data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01")
df_census_data.to_sql("CENSUS_DATA", con, if_exists="replace", index=False, method="multi")
df_chicago_public_schools = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01")
df_chicago_public_schools.to_sql("CHICAGO_PUBLIC_SCHOOLS", con, if_exists="replace", index=False, method="multi")
df_chicago_crime_data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCrimeData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01")
df_chicago_crime_data.to_sql("CHICAGO_CRIME_DATA", con, if_exists="replace", index=False, method="multi")

533

Hier wird die Brücke zwischen der Jupyter-Notebook-Umgebung (SQL-Magic) und der SQLite-Datenbank `FinalDB.db` aktiviert, um direkte Abfragen zu ermöglichen.


In [5]:
%sql sqlite:///FinalDB.db
# %sql: sql magic Befehl
# sqlite://: Das sagt dem Modul, welche Art von Datenbank du nutzt (den Datenbank-Treiber).
# / (der dritte Schrägstrich): Steht für den relativen Dateipfad. Er sagt: „Suche die Datei direkt hier im selben Ordner, in dem auch mein Jupyter Notebook liegt.“

Um einen ersten Überblick über die Datenstruktur zu erhalten, lasse ich mir im Folgenden die Details der Spalten der Tabellen anzeigen und werfe einen Blick auf die jeweils ersten drei Zeilen der Datensätze.

In [6]:
%sql PRAGMA table_info([CHICAGO_CRIME_DATA])

 * sqlite:///FinalDB.db
Done.


cid,name,type,notnull,dflt_value,pk
0,ID,INTEGER,0,None,0
1,CASE_NUMBER,TEXT,0,None,0
2,DATE,TEXT,0,None,0
3,BLOCK,TEXT,0,None,0
4,IUCR,TEXT,0,None,0
5,PRIMARY_TYPE,TEXT,0,None,0
6,DESCRIPTION,TEXT,0,None,0
7,LOCATION_DESCRIPTION,TEXT,0,None,0
8,ARREST,INTEGER,0,None,0
9,DOMESTIC,INTEGER,0,None,0


In [7]:
%sql PRAGMA table_info([CHICAGO_PUBLIC_SCHOOLS])

 * sqlite:///FinalDB.db
Done.


cid,name,type,notnull,dflt_value,pk
0,School_ID,INTEGER,0,None,0
1,NAME_OF_SCHOOL,TEXT,0,None,0
2,"Elementary, Middle, or High School",TEXT,0,None,0
3,Street_Address,TEXT,0,None,0
4,City,TEXT,0,None,0
5,State,TEXT,0,None,0
6,ZIP_Code,INTEGER,0,None,0
7,Phone_Number,TEXT,0,None,0
8,Link,TEXT,0,None,0
9,Network_Manager,TEXT,0,None,0


In [8]:
%sql PRAGMA table_info([CENSUS_DATA])

 * sqlite:///FinalDB.db
Done.


cid,name,type,notnull,dflt_value,pk
0,COMMUNITY_AREA_NUMBER,REAL,0,None,0
1,COMMUNITY_AREA_NAME,TEXT,0,None,0
2,PERCENT_OF_HOUSING_CROWDED,REAL,0,None,0
3,PERCENT_HOUSEHOLDS_BELOW_POVERTY,REAL,0,None,0
4,PERCENT_AGED_16__UNEMPLOYED,REAL,0,None,0
5,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,REAL,0,None,0
6,PERCENT_AGED_UNDER_18_OR_OVER_64,REAL,0,None,0
7,PER_CAPITA_INCOME,INTEGER,0,None,0
8,HARDSHIP_INDEX,REAL,0,None,0


In [9]:
%sql SELECT * FROM CHICAGO_CRIME_DATA LIMIT 3

 * sqlite:///FinalDB.db
Done.


ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION
3512276,HK587712,2004-08-28,047XX S KEDZIE AVE,890,THEFT,FROM BUILDING,SMALL RETAIL STORE,0,0,911,9,14.0,58.0,6,1155838.0,1873050.0,2004,41.8074405,-87.70395585,"(41.8074405, -87.703955849)"
3406613,HK456306,2004-06-26,009XX N CENTRAL PARK AVE,820,THEFT,$500 AND UNDER,OTHER,0,0,1112,11,27.0,23.0,6,1152206.0,1906127.0,2004,41.89827996,-87.71640551,"(41.898279962, -87.716405505)"
8002131,HT233595,2011-04-04,043XX S WABASH AVE,820,THEFT,$500 AND UNDER,NURSING HOME/RETIREMENT HOME,0,0,221,2,3.0,38.0,6,1177436.0,1876313.0,2011,41.81593313,-87.62464213,"(41.815933131, -87.624642127)"


In [10]:
%sql SELECT * FROM CHICAGO_PUBLIC_SCHOOLS LIMIT 3

 * sqlite:///FinalDB.db
Done.


School_ID,NAME_OF_SCHOOL,"Elementary, Middle, or High School",Street_Address,City,State,ZIP_Code,Phone_Number,Link,Network_Manager,Collaborative_Name,Adequate_Yearly_Progress_Made_,Track_Schedule,CPS_Performance_Policy_Status,CPS_Performance_Policy_Level,HEALTHY_SCHOOL_CERTIFIED,Safety_Icon,SAFETY_SCORE,Family_Involvement_Icon,Family_Involvement_Score,Environment_Icon,Environment_Score,Instruction_Icon,Instruction_Score,Leaders_Icon,Leaders_Score,Teachers_Icon,Teachers_Score,Parent_Engagement_Icon,Parent_Engagement_Score,Parent_Environment_Icon,Parent_Environment_Score,AVERAGE_STUDENT_ATTENDANCE,Rate_of_Misconducts__per_100_students_,Average_Teacher_Attendance,Individualized_Education_Program_Compliance_Rate,Pk_2_Literacy__,Pk_2_Math__,Gr3_5_Grade_Level_Math__,Gr3_5_Grade_Level_Read__,Gr3_5_Keep_Pace_Read__,Gr3_5_Keep_Pace_Math__,Gr6_8_Grade_Level_Math__,Gr6_8_Grade_Level_Read__,Gr6_8_Keep_Pace_Math_,Gr6_8_Keep_Pace_Read__,Gr_8_Explore_Math__,Gr_8_Explore_Read__,ISAT_Exceeding_Math__,ISAT_Exceeding_Reading__,ISAT_Value_Add_Math,ISAT_Value_Add_Read,ISAT_Value_Add_Color_Math,ISAT_Value_Add_Color_Read,Students_Taking__Algebra__,Students_Passing__Algebra__,9th Grade EXPLORE (2009),9th Grade EXPLORE (2010),10th Grade PLAN (2009),10th Grade PLAN (2010),Net_Change_EXPLORE_and_PLAN,11th Grade Average ACT (2011),Net_Change_PLAN_and_ACT,College_Eligibility__,Graduation_Rate__,College_Enrollment_Rate__,COLLEGE_ENROLLMENT,General_Services_Route,Freshman_on_Track_Rate__,X_COORDINATE,Y_COORDINATE,Latitude,Longitude,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,Ward,Police_District,Location
610038,Abraham Lincoln Elementary School,ES,615 W Kemper Pl,Chicago,IL,60614,(773) 534-5720,http://schoolreports.cps.edu/SchoolProgressReport_Eng/Spring2011Eng_610038.pdf,Fullerton Elementary Network,NORTH-NORTHWEST SIDE COLLABORATIVE,No,Standard,Not on Probation,Level 1,Yes,Very Strong,99.0,Very Strong,99,Strong,74.0,Strong,66.0,Weak,65,Strong,70,Strong,56,Average,47,96.00%,2.0,96.40%,95.80%,80.1,43.3,89.6,84.9,60.7,62.6,81.9,85.2,52,62.4,66.3,77.9,69.7,64.4,0.2,0.9,Yellow,Green,67.1,54.5,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,813,33,NDA,1171699.458,1915829.428,41.92449696,-87.64452163,7,LINCOLN PARK,43,18,"(41.92449696, -87.64452163)"
610281,Adam Clayton Powell Paideia Community Academy Elementary School,ES,7511 S South Shore Dr,Chicago,IL,60649,(773) 535-6650,http://schoolreports.cps.edu/SchoolProgressReport_Eng/Spring2011Eng_610281.pdf,Skyway Elementary Network,SOUTH SIDE COLLABORATIVE,No,Track_E,Not on Probation,Level 1,No,Average,54.0,Strong,66,Strong,74.0,Very Strong,84.0,Weak,63,Strong,76,Weak,46,Average,50,95.60%,15.7,95.30%,100.00%,62.4,51.7,21.9,15.1,29,42.8,38.5,27.4,44.8,42.7,14.1,34.4,16.8,16.5,0.7,1.4,Green,Green,17.2,27.3,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,521,46,NDA,1196129.985,1856209.466,41.76032435,-87.55673627,43,SOUTH SHORE,7,4,"(41.76032435, -87.55673627)"
610185,Adlai E Stevenson Elementary School,ES,8010 S Kostner Ave,Chicago,IL,60652,(773) 535-2280,http://schoolreports.cps.edu/SchoolProgressReport_Eng/Spring2011Eng_610185.pdf,Midway Elementary Network,SOUTHWEST SIDE COLLABORATIVE,No,Standard,Not on Probation,Level 2,No,Strong,61.0,NDA,NDA,Average,50.0,Weak,36.0,Weak,NDA,NDA,NDA,Average,47,Weak,41,95.70%,2.3,94.70%,98.30%,53.7,26.6,38.3,34.7,43.7,57.3,48.8,39.2,46.8,44,7.5,21.9,18.3,15.5,-0.9,-1.0,Red,Red,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,NDA,1324,44,NDA,1148427.165,1851012.215,41.74711093,-87.73170248,70,ASHBURN,13,8,"(41.74711093, -87.73170248)"


In [11]:
%sql SELECT * FROM CENSUS_DATA LIMIT 3

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0


## Mögliche Abfragen, die in der Praxis vorkommen

Nun können wir wie gewohnt SQL-Befehle eingeben, die zu möglichen Problemstellungen in der Realität auftauchen:

### Abfrage 1

##### Wie viele kriminelle Aktivitäten haben wir derzeit insgesamt erfasst?


In [12]:
%sql SELECT count(*) FROM CHICAGO_CRIME_DATA 

 * sqlite:///FinalDB.db
Done.


count(*)
533


### Abfrage 2

##### Welche Stadtteile (Community Areas) weisen ein Pro-Kopf-Einkommen zwischen 8000 und 10.500 USD auf?

In [13]:
%sql SELECT COMMUNITY_AREA_NAME, PER_CAPITA_INCOME FROM CENSUS_DATA WHERE PER_CAPITA_INCOME BETWEEN 8000 AND 10500

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,PER_CAPITA_INCOME
South Lawndale,10402
Fuller Park,10432
Riverdale,8201


### Abfrage 3

##### Welche Kriminalfälle betreffen Minderjährige? 


In [14]:
%sql SELECT CASE_NUMBER, DESCRIPTION FROM CHICAGO_CRIME_DATA WHERE DESCRIPTION LIKE '%MINOR%'

 * sqlite:///FinalDB.db
Done.


CASE_NUMBER,DESCRIPTION
HL266884,SELL/GIVE/DEL LIQUOR TO MINOR
HK238408,ILLEGAL CONSUMPTION BY MINOR


### Abfrage 4

##### Liste alle Körperverletzungen in Schulen auf.


In [15]:
%%sql 
SELECT PRIMARY_TYPE, DESCRIPTION, LOCATION_DESCRIPTION FROM CHICAGO_CRIME_DATA 
WHERE PRIMARY_TYPE LIKE '%battery%' AND LOCATION_DESCRIPTION LIKE '%school%'

 * sqlite:///FinalDB.db
Done.


PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION
BATTERY,SIMPLE,"SCHOOL, PUBLIC, GROUNDS"
BATTERY,PRO EMP HANDS NO/MIN INJURY,"SCHOOL, PUBLIC, BUILDING"
BATTERY,SIMPLE,"SCHOOL, PUBLIC, BUILDING"
BATTERY,SIMPLE,"SCHOOL, PUBLIC, BUILDING"
BATTERY,SIMPLE,"SCHOOL, PUBLIC, GROUNDS"


### Abfrage 5

##### Liste alle kriminellen Aktivitäten in Schulen auf ohne Körperverletzungen.


In [16]:
%%sql 
SELECT PRIMARY_TYPE, DESCRIPTION, LOCATION_DESCRIPTION FROM CHICAGO_CRIME_DATA 
WHERE LOCATION_DESCRIPTION LIKE '%school%' AND PRIMARY_TYPE <> 'BATTERY'

 * sqlite:///FinalDB.db
Done.


PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION
CRIMINAL DAMAGE,TO VEHICLE,"SCHOOL, PUBLIC, GROUNDS"
NARCOTICS,POSS: HEROIN(WHITE),"SCHOOL, PUBLIC, GROUNDS"
NARCOTICS,MANU/DEL:CANNABIS 10GM OR LESS,"SCHOOL, PUBLIC, BUILDING"
ASSAULT,PRO EMP HANDS NO/MIN INJURY,"SCHOOL, PUBLIC, GROUNDS"
CRIMINAL TRESPASS,TO LAND,"SCHOOL, PUBLIC, GROUNDS"
PUBLIC PEACE VIOLATION,BOMB THREAT,"SCHOOL, PRIVATE, BUILDING"
PUBLIC PEACE VIOLATION,BOMB THREAT,"SCHOOL, PUBLIC, BUILDING"


### Abfrage 6

##### Welche Schulform ist durchschnittlich am sichersten?


In [17]:
%%sql 
SELECT `Elementary, Middle, or High School`, AVG(SAFETY_SCORE) AS `Average_SAFETY_SCORE` FROM CHICAGO_PUBLIC_SCHOOLS 
GROUP BY `Elementary, Middle, or High School` ORDER BY Average_SAFETY_SCORE DESC LIMIT 1

 * sqlite:///FinalDB.db
Done.


"Elementary, Middle, or High School",Average_SAFETY_SCORE
HS,49.62352941176471


### Abfrage 7

##### Welche 3 Stadtteile sind am stärksten von Armut betroffen?


In [18]:
%%sql 
SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY FROM CENSUS_DATA
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC LIMIT 3

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6


### Abfrage 8

##### Welcher Stadtteil (Community Area) verzeichnet die geringste Kriminalität? (Als Ergebnis lasse ich mir ausschließlich die entsprechende Gebietsnummer ausgeben.)


In [19]:
%%sql 
SELECT COMMUNITY_AREA_NUMBER FROM CHICAGO_CRIME_DATA
WHERE COMMUNITY_AREA_NUMBER IS NOT NULL 
GROUP BY COMMUNITY_AREA_NUMBER ORDER BY count(ID) ASC LIMIT 1

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER
12.0


### Abfrage 9

##### Welcher Stadtteil hat den niedrigsten Härteindex (Hardship Index)?


In [20]:
%sql SELECT COMMUNITY_AREA_NAME FROM CENSUS_DATA WHERE HARDSHIP_INDEX=(SELECT MIN(HARDSHIP_INDEX) FROM CENSUS_DATA)

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Near North Side


### Abfrage 10

##### Welcher Stadtteil (Community Area) ist statistisch gesehen der sicherste?


In [21]:
%%sql 
SELECT COMMUNITY_AREA_NAME, count(ID) FROM CENSUS_DATA A, CHICAGO_CRIME_DATA B 
WHERE A.COMMUNITY_AREA_NUMBER = B.COMMUNITY_AREA_NUMBER
GROUP BY COMMUNITY_AREA_NAME ORDER BY count(ID) ASC LIMIT 1

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,count(ID)
Bridgeport,1
